# LLM Evaluation Protocols

This notebook demonstrates **Module 2** of the capstone workflow: preprocessing, protocol-specific data representations, few-shot prompt construction, structured output parsing, evaluation metrics, and failure-mode stress tests.

The reusable implementation has been moved to `src/llm/`. The original experimental workflow was **manual**: the generated prompt was run in ChatGPT and the returned JSON was pasted back into the notebook for parsing.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.preprocessing import preclean
from src.llm.serialization import standardize_input_representation
from src.llm.prompts import (
    FEW_SHOT_A, FEW_SHOT_B1, FEW_SHOT_B2, FEW_SHOT_C, FEW_SHOT_D,
    build_prompt,
)
from src.llm.parsing import parse_llm_output_module2
from src.llm.runner import run_single_replicate
from src.llm.metrics import point_estimation, compute_uncertainty
from src.llm.stress_tests import (
    trap_1, trap_2, trap_3,
    check_warning, check_stabilization_suggestion, interval_width,
)


## Protocols

- **A** — treated/control group summaries for difference-in-means
- **B1** — microdata for regression adjustment
- **B2** — compressed externally-computed regression output
- **C** — propensity-score bin summaries
- **D** — AIPW nuisance-prediction table

Use `standardize_input_representation(...)` to produce the appropriate representation, then `build_prompt(...)` to construct the ICL prompt.

In [ ]:
# Example (replace with a CSV already stored in your repository)
# csv_path = PROJECT_ROOT / "data" / "simulated" / "Engine_1_tau_1.csv"
# serialized = standardize_input_representation(csv_path, protocol="A")
# prompt = build_prompt(serialized, protocol="A", few_shot_examples=FEW_SHOT_A)
# print(prompt)


## Structured output parsing

`parse_llm_output_module2()` validates the required JSON keys, extracts ATE/CI/overlap fields, and checks the constraint `CI lower ≤ ATE ≤ CI upper` when all three values are numeric.

In [ ]:
# Example:
# parsed = parse_llm_output_module2(raw_json_string)
# parsed


## Evaluation and stress tests

`point_estimation()` reports bias and RMSE over valid point estimates.  
`compute_uncertainty()` reports CI coverage, average interval width, and the missing-CI rate.

The stress-test helpers probe post-treatment adjustment, instruments/strong treatment predictors, and weak-overlap behavior.